In [3]:
!pip install -q transformers langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 38.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [4]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
from tqdm import tqdm

In [5]:
import os

# List all datasets mounted under /kaggle/input
for dirname, _, filenames in os.walk('/kaggle/input'):
    print(f"Folder: {dirname}")
    for filename in filenames:
        print(f"  File: {filename}")


Folder: /kaggle/input
Folder: /kaggle/input/review-full
  File: reviews.csv


In [6]:
#import the reviews
df = pd.read_csv("/kaggle/input/review-full/reviews.csv")

In [7]:
df = df.dropna(subset=["comments"])
df.head(15)

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,68398,2894700,2012-11-16,3534183,Laura,I enjoyed my stay very much! Spend approx 6 we...
1,68398,13416072,2014-05-27,12750484,Maggie,I ended up staying in Studio B. Apartment is c...
2,68398,22427468,2014-11-06,21131563,Vincent,"Lisa est très efficace, le logement est foncti..."
3,68398,48514756,2015-09-27,31102107,Romney,I arrived a little early but Vincent valiantly...
4,68398,58001272,2015-12-31,44450239,Sheila,Our host made us feel very welcomed and was al...
5,68398,117113566,2016-12-04,26120062,Kathryn,"Awesome location. Cute, clean and stocked wit..."
6,68398,431471395,2019-04-01,126013182,Ha Neul,"Fair rate, clean room, comfortable bed, nice l..."
7,68398,984403762941767191,2023-09-20,470164880,Gita,Very nice!
8,68398,1013442366666899296,2023-10-30,209002431,Jan,"Wonderful stay! Can recommend, I felt right at..."
9,68398,1042475865706152726,2023-12-09,99035052,Amy,The location of this unit is at the popular Ea...


In [8]:
reviews = df["comments"]
reviews.head(15)
reviews.shape

(203800,)

In [9]:
#define the model
model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

model = model.to("cuda")

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

2025-07-12 12:50:57.099318: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752324657.278652      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752324657.332928      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

In [10]:
#Define sentiment scoring function

def get_sentiment_score(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to("cuda")
    with torch.no_grad(): #not in training model
        outputs = model(**inputs) #unpack inputs (a dictionary) and run the input text through the model
    probs = F.softmax(outputs.logits, dim=1)
    scores = torch.arange(1,6).float().to("cuda")
    weighted_avg = torch.sum(probs * scores, dim=1).item()
    return weighted_avg

    
        

In [ ]:
tqdm.pandas()
df["sentiment score"] = df["comments"].progress_apply(get_sentiment_score)

 31%|███▏      | 64162/203800 [08:59<19:06, 121.79it/s]

In [ ]:
df["score_1_to_10"] = df["sentiment score"] * 2

In [ ]:
df.to_csv("reviews_with_sentiment.csv", index=False)


In [ ]:
df.head()

In [ ]:
df_trimmed = df[["comments", "sentiment score", "score_1_to_10"]]

In [ ]:
df_trimmed.head(15)

In [ ]:
raw_score = df["sentiment score"]
adjusted_scored = df["score_1_to_10"]

In [ ]:
df_trimmed[df["sentiment score"] < 2]

In [ ]:
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

def detect_language(text):
    try:
        return detect(text)
    except:
        return "unknown"

In [ ]:
df["language"] = df["comments"].progress_apply(detect_language)

In [ ]:
df_trimmed = df[["listing_id","comments", "sentiment score", "score_1_to_10", "language"]]

In [ ]:
df_trimmed

In [ ]:
language_counts = df["language"].value_counts()
language_counts.index

In [ ]:
plt.figure(figsize=(8, 8))
plt.pie(language_counts.values,
        labels=language_counts.index,
        autopct='%1.1f%%',
        startangle=140)
plt.title("Languages used in reviews")
plt.axis('equal')
plt.show()

In [ ]:
!pip install plotly

import plotly.express as px

fig = px.pie(df, names="language", title="Languages Used in Reviews")
fig.show()


In [ ]:
df_trimmed